# Stage 3.4 — fine-tune YOLOv8n / YOLO11n / YOLO26n on the merged indoor set

Objective 1's joint accuracy/latency/deployment-cost benchmark needs all three
detectors trained with **identical hyperparameters** so the comparison is fair.
Policy from `CLAUDE.md` section 3: **no vertical flip, no mosaic** — a
chest-held phone never sees an upside-down or 4-image-grid viewpoint.

**Unverified — never executed.** This notebook is written against the real
output format `tools/prepare_data.py` produces (verified against actual
downloaded COCO + Roboflow data, 2026-09-07), but has not itself been run on
a GPU. Running it top-to-bottom on a Colab T4 is the actual verification
step — treat any error here as a version nudge, not a sign the approach is
wrong (same posture as the rest of this project's harness).

In [ ]:
!pip install -q ultralytics pandas

## 1. Get the merged dataset onto this Colab runtime

Zip `data/processed/merged` (contains `images/`, `labels/`, `provenance.csv`)
in the local repo and upload it here as `merged.zip` via the Colab file
browser, or change `MERGED_ZIP` to a Google-Drive-mounted path. Either way
this hand-off is manual — Colab has no access to the local Mac's filesystem.

In [ ]:
import os
from pathlib import Path

MERGED_ZIP = "/content/merged.zip"  # upload data/processed/merged.zip here, or point at Drive
DATASET_DIR = Path("/content/dataset")

if not DATASET_DIR.exists():
    assert os.path.exists(MERGED_ZIP), f"{MERGED_ZIP} not found -- upload a zip of data/processed/merged first"
    DATASET_DIR.mkdir(parents=True)
    !unzip -q "{MERGED_ZIP}" -d "{DATASET_DIR}"

print(sorted(os.listdir(DATASET_DIR)))

## 2. Build train/val file lists and `data.yaml` from `provenance.csv`

The `split` column is the scene-aware split `prepare_data.py` already
computed (grouped by original photo, not by frame — see CLAUDE.md section
10.3 for how that was verified against the real Roboflow export). Rows with
split `test_zeroshot` are COCO's zero-shot baseline and are deliberately
excluded from both training and validation here.

In [ ]:
import csv

# Keep this in sync with tools/taxonomy.py's CANONICAL_CLASSES by hand --
# duplicated here because this notebook runs on Colab without the local repo.
CANONICAL_CLASSES = [
    "person", "chair", "table", "door", "door_glass", "window",
    "stairs", "pole", "handrail", "wall", "low_obstacle", "other",
]

images_dir = DATASET_DIR / "images"
provenance_path = DATASET_DIR / "provenance.csv"

train_files, val_files = [], []
with open(provenance_path) as f:
    for row in csv.DictReader(f):
        split = row["split"]
        image_path = images_dir / row["new_filename"]
        if split == "train":
            train_files.append(str(image_path))
        elif split == "val":
            val_files.append(str(image_path))
        # test_zeroshot and anything else: not used for training/validation here

print(f"train: {len(train_files)}  val: {len(val_files)}")
assert train_files and val_files, "empty train or val split -- check provenance.csv's split column"

train_txt = DATASET_DIR / "train.txt"
val_txt = DATASET_DIR / "val.txt"
train_txt.write_text("\n".join(train_files) + "\n")
val_txt.write_text("\n".join(val_files) + "\n")

data_yaml = DATASET_DIR / "data.yaml"
data_yaml.write_text(
    f"train: {train_txt}\n"
    f"val: {val_txt}\n"
    f"nc: {len(CANONICAL_CLASSES)}\n"
    f"names: {CANONICAL_CLASSES}\n"
)
print(data_yaml.read_text())

## 3. Shared hyperparameters — identical across all three detectors

YOLO26 is natively NMS-free (CLAUDE.md section 6). Ultralytics' `train()`/
`val()` API doesn't expose a separate NMS toggle at training time for any of
these three models either way, so no hyperparameter below needs to differ
for it — but don't be surprised if its `val()` metrics object shapes up
slightly differently, and don't add an NMS-related kwarg for it if a future
Ultralytics version does add one.

In [ ]:
COMMON_TRAIN_KWARGS = dict(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    fliplr=0.5,    # horizontal flip -- a chest-held phone sees both left- and right-facing scenes
    flipud=0.0,    # NO vertical flip -- the phone is never upside down
    degrees=10.0,  # mild rotation -- some hand tilt, not tumbling
    translate=0.1,
    scale=0.2,
    shear=0.0,
    perspective=0.0,
    mosaic=0.0,    # disabled -- mosaic teaches 4-image grid layouts the camera never sees
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,     # modest brightness/colour jitter
    verbose=True,
)
# Ultralytics' built-in Albumentations pipeline (Blur/MedianBlur/ToGray/CLAHE) fires at its
# own small fixed probability regardless of the kwargs above -- there is no train() parameter
# exposed to tune "blur" specifically; this is the closest available knob to that requirement.

MODELS = ["yolov8n.pt", "yolo11n.pt", "yolo26n.pt"]
# yolo26n.pt may not resolve on older ultralytics releases -- if this fails, upgrade
# ultralytics rather than substituting a different model (see CLAUDE.md section 6).

## 4. Train and evaluate each detector, saving per-class AP

Per-class AP alongside the pooled mAP, not instead of it — CLAUDE.md section
5: a single pooled mAP can hide an undertrained high-consequence class like
`stairs`.

In [ ]:
import shutil
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

WEIGHTS_DIR = Path("/content/weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

all_results = []

for weights in MODELS:
    stem = weights.replace(".pt", "")
    print(f"\n=== training {stem} ===")
    model = YOLO(weights)
    model.train(name=stem, **COMMON_TRAIN_KWARGS)
    metrics = model.val(data=str(data_yaml), imgsz=640, split="val")

    # Save the fine-tuned weights -- these, not the pretrained COCO checkpoints,
    # are what export_coreml.py needs for Stage 4.
    best_pt = model.trainer.save_dir / "weights" / "best.pt"
    shutil.copy(best_pt, WEIGHTS_DIR / f"{stem}_best.pt")

    box = metrics.box
    for class_idx, ap50_95, ap50 in zip(box.ap_class_index, box.ap, box.ap50):
        all_results.append({
            "model": stem,
            "class": CANONICAL_CLASSES[int(class_idx)],
            "ap50_95": float(ap50_95),
            "ap50": float(ap50),
        })
    all_results.append({
        "model": stem,
        "class": "__pooled_mAP__",
        "ap50_95": float(box.map),
        "ap50": float(box.map50),
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("/content/objective1_train_metrics.csv", index=False)
results_df

## 5. Download the results CSV *and* the fine-tuned weights back to the local repo

Save the CSV as `results/objective1_train_metrics.csv`. Save the weights zip
somewhere under `data/` (gitignored) -- `export_coreml.py --weights-dir <dir>`
expects files named `yolov8n.pt`/`yolo11n.pt`/`yolo26n.pt`, so rename the
extracted `<stem>_best.pt` files accordingly (drop the `_best` suffix) before
pointing it there.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/finetuned_weights", "zip", WEIGHTS_DIR)
files.download("/content/objective1_train_metrics.csv")
files.download("/content/finetuned_weights.zip")